# 01 — Source Database Structure Profile

## Purpose

This notebook profiles the supplied SQLite source database:

`data/raw/form_2015-present/form_2015-present/raceform.db`

The purpose is to document the database as supplied, before any cleaning, transformation, integration or target-schema design.

The notebook will inspect:

- file validity and basic metadata;
- SQLite metadata and relevant pragmas;
- tables and views;
- declared schemas and data types;
- row counts;
- declared primary keys and foreign keys;
- indexes;
- approximate date coverage;
- likely table grain;
- apparent relationships;
- fields requiring deeper racing-domain investigation.

## Evidence categories

Findings in this notebook will be classified as:

1. **Source declaration** — information explicitly declared in SQLite metadata or schema definitions.
2. **Profiling evidence** — information demonstrated by queries against the supplied database.
3. **Initial hypothesis** — a provisional interpretation that requires further technical or racing-domain investigation.

A plausible table name, column name or relationship is not treated as established fact without supporting evidence.

## Boundaries

This notebook will not:

- clean or standardise source values;
- remove or consolidate rows;
- infer missing values;
- redesign the source schema;
- select the final database platform;
- combine `raceform.db` with any other supplied data product;
- create production tables or analytical features.

All access to the source database should be read-only wherever practicable.

In [1]:
from pathlib import Path
import os
import sqlite3
import sys

import pandas as pd
from IPython.display import display

# Locate the project root whether the notebook is started from the repository
# root or from the notebooks directory.
working_directory = Path.cwd().resolve()

if (working_directory / "data").exists():
    project_root = working_directory
elif (working_directory.parent / "data").exists():
    project_root = working_directory.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. Expected a data/ directory in "
        f"{working_directory} or {working_directory.parent}."
    )

source_db_path = (
    project_root
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

print(f"Project root:   {project_root}")
print(f"Source database: {source_db_path}")
print(f"File exists:     {source_db_path.is_file()}")

Project root:   /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
File exists:     True


## 1. Source File Verification

Before opening the database, this section inspects the source file as a filesystem object.

The checks establish:

- whether the expected path points to a regular file;
- the file size;
- filesystem timestamps;
- access permissions;
- whether the first bytes contain the standard SQLite database header.

The timestamps are filesystem metadata and do not necessarily represent when the racing data was created or last updated.

A valid SQLite header provides evidence that the file has the expected format, but it does not by itself prove that the complete database is internally valid.

In [3]:
from datetime import datetime


def format_timestamp(timestamp: float) -> str:
    """
    Convert a Unix filesystem timestamp into a readable local-time string.

    The timezone is taken from the machine running the notebook.
    """
    return datetime.fromtimestamp(timestamp).astimezone().isoformat(
        sep=" ",
        timespec="seconds",
    )


# Read filesystem metadata without opening the file as a SQLite database.
file_stat = source_db_path.stat()

# Read only the first 16 bytes of the file.
# A standard SQLite 3 database begins with this fixed header.
with source_db_path.open("rb") as source_file:
    sqlite_header = source_file.read(16)

expected_sqlite_header = b"SQLite format 3\x00"

# Record the initial file-level checks in a compact, readable table.
# These observations come from the filesystem and file contents, not from
# SQLite metadata or SQL queries.
file_metadata = pd.DataFrame(
    [
        {
            "property": "Resolved path",
            "value": str(source_db_path.resolve()),
        },
        {
            "property": "Regular file",
            "value": source_db_path.is_file(),
        },
        {
            "property": "File size (bytes)",
            "value": file_stat.st_size,
        },
        {
            "property": "File size (MiB)",
            "value": round(file_stat.st_size / (1024**2), 2),
        },
        {
            "property": "Last modified",
            "value": format_timestamp(file_stat.st_mtime),
        },
        {
            "property": "Metadata changed",
            "value": format_timestamp(file_stat.st_ctime),
        },
        {
            "property": "Readable",
            "value": os.access(source_db_path, os.R_OK),
        },
        {
            "property": "Writable",
            "value": os.access(source_db_path, os.W_OK),
        },
        {
            "property": "First 16 bytes",
            "value": repr(sqlite_header),
        },
        {
            "property": "Standard SQLite header",
            "value": sqlite_header == expected_sqlite_header,
        },
    ]
)

display(file_metadata)

,property,value
0,Resolved path,/home/rob/Documents/inside-rails-horse-racing/...
1,Regular file,True
2,File size (bytes),765825024
3,File size (MiB),730.35
4,Last modified,2026-06-04 06:09:24+01:00
5,Metadata changed,2026-07-21 13:08:30+01:00
6,Readable,True
7,Writable,True
8,First 16 bytes,b'SQLite format 3\x00'
9,Standard SQLite header,True


### Initial file-level findings

**Profiling evidence**

- The expected source path points to a regular file.
- The file is 765,825,024 bytes in size, approximately 730.35 MiB.
- The first 16 bytes match the standard SQLite 3 file header.
- The file is readable by the current user.

**Operational observation**

The source file is also writable under the current filesystem permissions. This does not authorise modification. The notebook will open the database through SQLite's read-only connection mode to reduce the risk of accidental changes.

**Limitation**

The SQLite header confirms the expected file format, but it does not establish that every database page or object is internally valid. Database-level checks are still required.

In [4]:
# Convert the source path into a SQLite file URI.
# The query parameter mode=ro instructs SQLite to reject write operations.
source_db_uri = f"{source_db_path.resolve().as_uri()}?mode=ro"

try:
    # Open the supplied database explicitly in read-only mode.
    connection = sqlite3.connect(
        source_db_uri,
        uri=True,
    )

    # Confirm that SQLite can execute a basic query against the opened file.
    # sqlite_version() reports the SQLite library version used by Python,
    # not necessarily the version originally used to create the database.
    sqlite_version = connection.execute(
        "SELECT sqlite_version();"
    ).fetchone()[0]

    # Read SQLite's internal schema catalogue.
    # Successfully querying sqlite_schema demonstrates that SQLite can parse
    # the database catalogue, although it is not yet a full integrity test.
    schema_object_count = connection.execute(
        """
        SELECT COUNT(*)
        FROM sqlite_schema;
        """
    ).fetchone()[0]

    connection_status = pd.DataFrame(
        [
            {
                "check": "Read-only connection opened",
                "result": True,
            },
            {
                "check": "SQLite library version",
                "result": sqlite_version,
            },
            {
                "check": "Objects recorded in sqlite_schema",
                "result": schema_object_count,
            },
        ]
    )

    display(connection_status)

except sqlite3.Error as error:
    # Stop the notebook immediately if SQLite cannot open or read the file.
    raise RuntimeError(
        f"SQLite could not open the source database in read-only mode: {error}"
    ) from error

,check,result
0,Read-only connection opened,True
1,SQLite library version,3.45.1
2,Objects recorded in sqlite_schema,1


### Initial database-level findings

**Profiling evidence**

- SQLite successfully opened the file using explicit read-only mode.
- Python is using SQLite library version 3.45.1.
- The database catalogue contains one schema object.

**Initial hypothesis**

The source may contain one large table rather than a multi-table relational structure.

This remains a hypothesis until the object type and declared schema have been inspected.

In [5]:
# Query SQLite's schema catalogue for every user-visible database object.
#
# sqlite_schema records tables, views, indexes and triggers. Internal objects
# whose names begin with "sqlite_" are excluded here because they are created
# and managed by SQLite rather than supplied as part of the source design.
schema_inventory = pd.read_sql_query(
    """
    SELECT
        type AS object_type,
        name AS object_name,
        tbl_name AS associated_table,
        rootpage,
        sql AS declared_sql
    FROM sqlite_schema
    WHERE name NOT LIKE 'sqlite_%'
    ORDER BY
        CASE type
            WHEN 'table' THEN 1
            WHEN 'view' THEN 2
            WHEN 'index' THEN 3
            WHEN 'trigger' THEN 4
            ELSE 5
        END,
        name;
    """,
    connection,
)

# Display the full result because the catalogue currently contains only one
# object. Later inventories may use summary views where the output is larger.
display(schema_inventory)

,object_type,object_name,associated_table,rootpage,declared_sql
0,table,data,data,2,"CREATE TABLE data (\n date NUMERIC,\..."


### Schema-object inventory findings

**Source declaration**

- The database declares one user table named `data`.
- The table's root page is page 2.
- No user-defined views, indexes or triggers appear in the schema catalogue.

**Profiling evidence**

The complete user-visible schema inventory contains one object.

**Initial interpretation**

The supplied `raceform.db` file appears to be a single-table data product rather than a normalised relational database.

This describes the supplied structure only. It does not yet establish the table's grain, key structure or racing meaning.

In [6]:
# Escape any embedded quotation marks before placing the table name inside
# a SQLite identifier. The table name comes from SQLite's own schema catalogue,
# but quoting it remains good practice.
source_table_name = schema_inventory.loc[
    schema_inventory["object_type"].eq("table"),
    "object_name",
].iloc[0]

quoted_table_name = source_table_name.replace('"', '""')

# PRAGMA table_info reports the columns declared for the supplied table.
#
# The output describes the source schema as SQLite records it:
# - cid: ordinal position of the column;
# - name: declared column name;
# - type: declared SQLite data type;
# - notnull: whether NOT NULL was explicitly declared;
# - dflt_value: declared default expression;
# - pk: position within the declared primary key, where zero means the
#   column is not part of a declared primary key.
declared_columns = pd.read_sql_query(
    f'PRAGMA table_info("{quoted_table_name}");',
    connection,
)

# Rename SQLite's compact pragma fields so the notebook output is easier to
# interpret without changing any underlying values.
declared_columns = declared_columns.rename(
    columns={
        "cid": "column_position",
        "name": "column_name",
        "type": "declared_type",
        "notnull": "not_null_declared",
        "dflt_value": "declared_default",
        "pk": "primary_key_position",
    }
)

# Convert SQLite's integer flag into a Boolean display value.
declared_columns["not_null_declared"] = (
    declared_columns["not_null_declared"].astype(bool)
)

display(declared_columns)

,column_position,column_name,declared_type,not_null_declared,declared_default,primary_key_position
0,0,date,NUMERIC,False,None,0
1,1,course,TEXT,False,None,0
2,2,race_id,INTEGER,False,None,0
3,3,off,TEXT,False,None,0
4,4,race_name,TEXT,False,None,0
5,5,type,TEXT,False,None,0
6,6,class,TEXT,False,None,0
7,7,pattern,TEXT,False,None,0
8,8,rating_band,TEXT,False,None,0
9,9,age_band,TEXT,False,None,0


### Declared column-schema findings

**Source declaration**

The `data` table declares 37 columns:

- 20 columns with declared type `TEXT`;
- 14 columns with declared type `INTEGER`;
- 3 columns with declared type `NUMERIC`.

The source declares:

- no primary key;
- no `NOT NULL` constraints;
- no default values.

**Initial structural interpretation**

The column names appear to mix information about:

- the race, such as `date`, `course`, `race_id`, `race_name`, `class`, `dist` and `going`;
- the runner, such as `horse`, `draw`, `pos`, `jockey`, `trainer`, `owner` and ratings.

This suggests that each row may represent one runner in one race, with race-level attributes repeated across runners.

That remains an initial hypothesis. It must be tested through row counts, distinct identifiers and within-race consistency checks.

Column names are not yet treated as complete definitions of racing meaning. Fields such as `off`, `type`, `pattern`, `sex_rest`, `ovr_btn`, `btn`, `hg`, `or`, `rpr` and `ts` require later domain investigation.

In [7]:
# PRAGMA table_list provides table-level properties introduced in newer
# SQLite versions. It shows whether the object is a normal table, view,
# virtual table or shadow table, and whether special table options apply.
table_properties = pd.read_sql_query(
    "PRAGMA table_list;",
    connection,
)

# Restrict the result to the supplied user table. SQLite may also report
# internal catalogue tables such as sqlite_schema.
table_properties = table_properties.loc[
    table_properties["name"].eq(source_table_name)
].copy()

# Rename SQLite's abbreviated fields to make the output self-explanatory.
table_properties = table_properties.rename(
    columns={
        "schema": "database_schema",
        "name": "table_name",
        "type": "object_type",
        "ncol": "declared_column_count",
        "wr": "without_rowid",
        "strict": "strict_table",
    }
)

# SQLite returns integer flags for WITHOUT ROWID and STRICT.
# Convert them to Boolean values for clearer notebook output.
for flag_column in ["without_rowid", "strict_table"]:
    table_properties[flag_column] = (
        table_properties[flag_column].astype(bool)
    )

# Produce a compact summary of the column-level declarations already
# retrieved through PRAGMA table_info.
declared_schema_summary = pd.DataFrame(
    [
        {
            "property": "Declared columns",
            "value": len(declared_columns),
        },
        {
            "property": "Columns declaring NOT NULL",
            "value": int(
                declared_columns["not_null_declared"].sum()
            ),
        },
        {
            "property": "Columns in declared primary key",
            "value": int(
                declared_columns["primary_key_position"].gt(0).sum()
            ),
        },
        {
            "property": "Columns with declared defaults",
            "value": int(
                declared_columns["declared_default"].notna().sum()
            ),
        },
    ]
)

print("Table properties")
display(table_properties)

print("Declared constraint summary")
display(declared_schema_summary)

Table properties


,database_schema,table_name,object_type,declared_column_count,without_rowid,strict_table
0,main,data,table,37,False,False


Declared constraint summary


,property,value
0,Declared columns,37
1,Columns declaring NOT NULL,0
2,Columns in declared primary key,0
3,Columns with declared defaults,0


### Table-property and constraint findings

**Source declaration**

The `data` object is declared as a normal table in the `main` database schema.

It has:

- 37 declared columns;
- no declared primary key;
- no declared `NOT NULL` constraints;
- no declared default values;
- no `WITHOUT ROWID` option;
- no `STRICT` table option.

**Initial interpretation**

Because the table is not declared `WITHOUT ROWID`, SQLite provides an implicit `rowid` for storage purposes.

That implicit `rowid` must not be treated automatically as a racing-domain identifier or durable business key. It may only reflect the physical insertion order of rows in this particular database file.

In [8]:
# PRAGMA foreign_key_list reports foreign-key constraints declared for the
# table. An empty result means the source schema declares no foreign keys;
# it does not prove that no logical relationships exist in the data.
declared_foreign_keys = pd.read_sql_query(
    f'PRAGMA foreign_key_list("{quoted_table_name}");',
    connection,
)

# PRAGMA index_list reports indexes associated with the supplied table.
# This includes explicit indexes and SQLite-generated indexes supporting
# declared UNIQUE or PRIMARY KEY constraints.
declared_indexes = pd.read_sql_query(
    f'PRAGMA index_list("{quoted_table_name}");',
    connection,
)

# Build a compact summary while retaining the complete pragma outputs for
# inspection if any declared objects are found.
relational_declaration_summary = pd.DataFrame(
    [
        {
            "property": "Declared foreign keys",
            "value": len(declared_foreign_keys),
        },
        {
            "property": "Indexes associated with table",
            "value": len(declared_indexes),
        },
    ]
)

display(relational_declaration_summary)

# Display the detailed results only when SQLite reports declared foreign keys
# or indexes. This avoids showing empty tables without hiding any evidence.
if not declared_foreign_keys.empty:
    print("Declared foreign keys")
    display(declared_foreign_keys)
else:
    print("No foreign keys are declared for the data table.")

if not declared_indexes.empty:
    print("Associated indexes")
    display(declared_indexes)
else:
    print("No indexes are associated with the data table.")

,property,value
0,Declared foreign keys,0
1,Indexes associated with table,0


No foreign keys are declared for the data table.
No indexes are associated with the data table.


### Declared relational-structure findings

**Source declaration**

The `data` table has:

- no declared foreign keys;
- no associated indexes;
- no declared uniqueness constraints visible through SQLite's index catalogue.

**Interpretation**

The source schema does not formally define relationships, uniqueness or lookup performance.

This does not prove that identifiers such as `race_id` are invalid or that no logical relationships exist. Their behaviour must be tested through profiling.

In [9]:
# Count every row in the supplied table.
#
# This is the first content-level query against the source table. It does not
# clean, transform or reinterpret any values.
row_count = connection.execute(
    f'SELECT COUNT(*) FROM "{quoted_table_name}";'
).fetchone()[0]

# Inspect the minimum and maximum implicit rowid values.
#
# Because the table is a normal SQLite rowid table, each row has an internal
# rowid unless explicitly overridden. These values describe storage structure,
# not necessarily a stable or meaningful racing identifier.
rowid_summary = connection.execute(
    f"""
    SELECT
        MIN(rowid),
        MAX(rowid),
        COUNT(DISTINCT rowid)
    FROM "{quoted_table_name}";
    """
).fetchone()

table_row_summary = pd.DataFrame(
    [
        {
            "property": "Total rows",
            "value": row_count,
        },
        {
            "property": "Minimum rowid",
            "value": rowid_summary[0],
        },
        {
            "property": "Maximum rowid",
            "value": rowid_summary[1],
        },
        {
            "property": "Distinct rowids",
            "value": rowid_summary[2],
        },
        {
            "property": "Rowids are unique",
            "value": rowid_summary[2] == row_count,
        },
        {
            "property": "Rowid range is contiguous",
            "value": (
                rowid_summary[0] == 1
                and rowid_summary[1] == row_count
            ),
        },
    ]
)

display(table_row_summary)

,property,value
0,Total rows,1851286
1,Minimum rowid,1
2,Maximum rowid,1851286
3,Distinct rowids,1851286
4,Rowids are unique,True
5,Rowid range is contiguous,True


### Row-count and rowid findings

**Profiling evidence**

The `data` table contains 1,851,286 rows.

Its implicit SQLite `rowid` values:

- begin at 1;
- end at 1,851,286;
- are unique;
- form a contiguous sequence.

**Interpretation**

The contiguous `rowid` range is consistent with rows having been inserted sequentially without deletions in this supplied file.

The `rowid` is not declared by the source as a racing identifier and must not be treated as a stable business key.

In [11]:
# Profile race_id as a possible race-level identifier.
#
# This does not assume that race_id is valid or unique. The query measures:
# - how many rows have and do not have a race_id;
# - how many distinct non-null race_id values exist;
# - the smallest and largest observed values.
race_id_summary = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(race_id) AS rows_with_race_id,
        SUM(CASE WHEN race_id IS NULL THEN 1 ELSE 0 END) AS null_race_id_rows,
        COUNT(DISTINCT race_id) AS distinct_race_ids,
        MIN(race_id) AS minimum_race_id,
        MAX(race_id) AS maximum_race_id
    FROM "{quoted_table_name}";
    """,
    connection,
)

# Count how many runner-like rows are associated with each non-null race_id.
#
# If the table grain is one row per runner, most race_id values should appear
# several times because each race normally contains several runners.
rows_per_race_id = pd.read_sql_query(
    f"""
    SELECT
        race_id,
        COUNT(*) AS row_count
    FROM "{quoted_table_name}"
    WHERE race_id IS NOT NULL
    GROUP BY race_id;
    """,
    connection,
)

# Summarise the distribution without yet claiming that race_id is a valid key.
rows_per_race_id_summary = pd.DataFrame(
    [
        {
            "metric": "Minimum rows per race_id",
            "value": int(rows_per_race_id["row_count"].min()),
        },
        {
            "metric": "Median rows per race_id",
            "value": float(rows_per_race_id["row_count"].median()),
        },
        {
            "metric": "Mean rows per race_id",
            "value": round(rows_per_race_id["row_count"].mean(), 2),
        },
        {
            "metric": "Maximum rows per race_id",
            "value": int(rows_per_race_id["row_count"].max()),
        },
        {
            "metric": "race_id values appearing once",
            "value": int(rows_per_race_id["row_count"].eq(1).sum()),
        },
    ]
)

print("race_id coverage")
display(race_id_summary)

print("Rows associated with each non-null race_id")
display(rows_per_race_id_summary)

race_id coverage


,total_rows,rows_with_race_id,null_race_id_rows,distinct_race_ids,minimum_race_id,maximum_race_id
0,1851286,1851286,0,188783,0,race_id


Rows associated with each non-null race_id


,metric,value
0,Minimum rows per race_id,1.00
1,Median rows per race_id,9.00
2,Mean rows per race_id,9.81
3,Maximum rows per race_id,57.00
4,race_id values appearing once,23.00


### Initial `race_id` findings

**Profiling evidence**

- All 1,851,286 rows contain a non-null `race_id` value.
- There are 188,783 distinct `race_id` values.
- A typical `race_id` occurs across several rows:
  - median: 9 rows;
  - mean: 9.81 rows;
  - maximum: 57 rows.
- Twenty-three `race_id` values occur only once.

This distribution is consistent with, but does not yet prove, a runner-level table in which several rows belong to each race.

**Unexpected result**

Although `race_id` is declared as `INTEGER`, the reported maximum is the text value `race_id`.

Because the table is not `STRICT`, SQLite can store text in an integer-declared column. The actual storage classes and anomalous values must therefore be inspected before treating `race_id` as numeric.

In [12]:
# SQLite's typeof() function reports the storage class of each stored value,
# independently of the column's declared type.
#
# This check determines whether every race_id is actually stored as an integer
# or whether the non-STRICT table contains values of other storage classes.
race_id_storage_types = pd.read_sql_query(
    f"""
    SELECT
        typeof(race_id) AS sqlite_storage_type,
        COUNT(*) AS row_count
    FROM "{quoted_table_name}"
    GROUP BY typeof(race_id)
    ORDER BY row_count DESC;
    """,
    connection,
)

display(race_id_storage_types)

,sqlite_storage_type,row_count
0,integer,1851285
1,text,1


In [13]:
# Retrieve every row whose race_id is not stored as a SQLite integer.
#
# The selected context columns should help determine whether these are genuine
# racing records, repeated headers, import artefacts or another source pattern.
non_integer_race_ids = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        typeof(race_id) AS race_id_storage_type,
        race_id,
        date,
        course,
        off,
        race_name,
        horse,
        jockey,
        trainer
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) <> 'integer'
    ORDER BY rowid;
    """,
    connection,
)

print(f"Rows with a non-integer race_id: {len(non_integer_race_ids):,}")
display(non_integer_race_ids)

Rows with a non-integer race_id: 1


,source_rowid,race_id_storage_type,race_id,date,course,off,race_name,horse,jockey,trainer
0,1,text,race_id,date,course,off,race_name,horse,jockey,trainer


### `race_id` storage-type findings

**Profiling evidence**

- 1,851,285 rows store `race_id` as a SQLite integer.
- One row stores `race_id` as text.
- The non-integer row is `rowid = 1`.
- Its values repeat column names such as `date`, `course`, `race_id`, `off`, `race_name`, `horse`, `jockey` and `trainer`.

**Interpretation**

The first stored row appears to be a header row imported into the table as data.

This is an observed source-quality issue, not yet a cleaning decision. Notebook 01 will preserve and document the row rather than delete it.

The finding also demonstrates that declared SQLite types cannot be assumed to match every stored value in this non-`STRICT` table.

In [14]:
# Recalculate the race_id profile using only values that SQLite stores as
# integers. This avoids allowing the identified header artefact to distort
# numeric minimum, maximum and grouping statistics.
numeric_race_id_summary = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS rows_with_integer_race_id,
        COUNT(DISTINCT race_id) AS distinct_integer_race_ids,
        MIN(race_id) AS minimum_race_id,
        MAX(race_id) AS maximum_race_id
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer';
    """,
    connection,
)

# Recalculate rows per race_id after excluding the single non-integer value.
numeric_rows_per_race_id = pd.read_sql_query(
    f"""
    SELECT
        race_id,
        COUNT(*) AS row_count
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer'
    GROUP BY race_id;
    """,
    connection,
)

numeric_rows_per_race_id_summary = pd.DataFrame(
    [
        {
            "metric": "Minimum rows per race_id",
            "value": int(numeric_rows_per_race_id["row_count"].min()),
        },
        {
            "metric": "Median rows per race_id",
            "value": float(numeric_rows_per_race_id["row_count"].median()),
        },
        {
            "metric": "Mean rows per race_id",
            "value": round(
                numeric_rows_per_race_id["row_count"].mean(),
                2,
            ),
        },
        {
            "metric": "Maximum rows per race_id",
            "value": int(numeric_rows_per_race_id["row_count"].max()),
        },
        {
            "metric": "race_id values appearing once",
            "value": int(
                numeric_rows_per_race_id["row_count"].eq(1).sum()
            ),
        },
    ]
)

print("Integer race_id coverage")
display(numeric_race_id_summary)

print("Rows per integer race_id")
display(numeric_rows_per_race_id_summary)

Integer race_id coverage


,rows_with_integer_race_id,distinct_integer_race_ids,minimum_race_id,maximum_race_id
0,1851285,188782,0,4803590


Rows per integer race_id


,metric,value
0,Minimum rows per race_id,1.00
1,Median rows per race_id,9.00
2,Mean rows per race_id,9.81
3,Maximum rows per race_id,57.00
4,race_id values appearing once,22.00


### Numeric `race_id` findings

**Profiling evidence**

After excluding the single text header artefact from numeric summaries:

- 1,851,285 rows contain integer-stored `race_id` values;
- there are 188,782 distinct integer `race_id` values;
- the observed range is 0 to 4,803,590;
- the median number of rows per `race_id` is 9;
- the mean number of rows per `race_id` is 9.81;
- the maximum number of rows for one `race_id` is 57;
- 22 integer `race_id` values occur only once.

**Initial interpretation**

The repeated `race_id` pattern strongly supports the hypothesis that rows represent race participants rather than races themselves.

However, `race_id` has not yet been demonstrated to identify one internally consistent race. The zero value, singleton groups and within-group consistency require investigation.

In [15]:
# Count and inspect rows assigned to race_id = 0.
#
# Zero may be a valid source identifier, a missing-value substitute or an
# import artefact. The notebook records its observed context before assigning
# any meaning.
race_id_zero_count = connection.execute(
    f"""
    SELECT COUNT(*)
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer'
      AND race_id = 0;
    """
).fetchone()[0]

race_id_zero_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        class,
        dist,
        going,
        horse,
        pos,
        jockey,
        trainer
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer'
      AND race_id = 0
    ORDER BY rowid
    LIMIT 25;
    """,
    connection,
)

print(f"Rows with race_id = 0: {race_id_zero_count:,}")
display(race_id_zero_rows)

Rows with race_id = 0: 15


,source_rowid,date,course,race_id,off,race_name,type,class,dist,going,horse,pos,jockey,trainer
0,286607,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Loud And Clear (GB),1,Ross Chapman,Iain Jardine
1,286614,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Donnas Delight (IRE),2,Steven Fox,Sandy Thomson
2,286624,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Pot Committed (IRE),3,Shane Shortall,Iain Jardine
3,286625,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Carry On Arcadio (IRE),4,Brian Harding,Nicky Richards
4,286626,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Strong Economy (IRE),6,Stephen Mulqueen,R Mike Smith
5,286635,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,The Germany One (IRE),15,Matthew Bowes,Lady Jane Gillespie
6,286636,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Solway Storm (IRE),14,Callum Bewley,Lisa Harrison
7,286637,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Captain Sam (GB),13,Brian Hughes,Malcolm Jefferson
8,286638,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Mullaghboy (IRE),12,Richard Johnson,S R B Crawford
9,286639,2016-10-24,Ayr,0,4:40,Racing UK Day Pass Just £10 Standard Open Nati...,NH Flat,Class 6,2m,Good To Soft,Seelateralligator (IRE),11,Harry Skelton,Dan Skelton


In [16]:
# Test whether fields that appear race-level remain constant within each
# integer race_id.
#
# A race_id is flagged when a field has more than one distinct non-null value.
# This is an initial structural check, not a final business-rule validation.
race_level_consistency = pd.read_sql_query(
    f"""
    SELECT
        race_id,
        COUNT(*) AS row_count,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT course) AS distinct_courses,
        COUNT(DISTINCT off) AS distinct_off_times,
        COUNT(DISTINCT race_name) AS distinct_race_names,
        COUNT(DISTINCT type) AS distinct_types,
        COUNT(DISTINCT class) AS distinct_classes,
        COUNT(DISTINCT dist) AS distinct_distances,
        COUNT(DISTINCT going) AS distinct_going_values,
        COUNT(DISTINCT ran) AS distinct_ran_values
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer'
    GROUP BY race_id;
    """,
    connection,
)

# Summarise how many race_id groups contain conflicting values for each
# apparently race-level field.
consistency_summary = pd.DataFrame(
    [
        {
            "field": column.removeprefix("distinct_"),
            "race_ids_with_multiple_values": int(
                race_level_consistency[column].gt(1).sum()
            ),
        }
        for column in race_level_consistency.columns
        if column.startswith("distinct_")
    ]
)

display(consistency_summary)

,field,race_ids_with_multiple_values
0,dates,206
1,courses,207
2,off_times,210
3,race_names,210
4,types,92
5,classes,166
6,distances,198
7,going_values,179
8,ran_values,195


### `race_id = 0` findings

**Profiling evidence**

Fifteen rows have `race_id = 0`.

All fifteen rows describe runners in what appears to be the same race:

- date: 24 October 2016;
- course: Ayr;
- off time: 4:40;
- race type: NH Flat;
- class: Class 6;
- distance: 2m;
- going: Good To Soft.

The rows contain different horses, finishing positions, jockeys and trainers.

**Initial interpretation**

Within this source, `race_id = 0` appears to function as the identifier for one genuine race rather than as a general representation of missing data.

The value may still indicate an identifier-generation problem or exceptional source case, but it must not be converted to missing or removed merely because it is zero.

### Initial within-`race_id` consistency findings

**Profiling evidence**

Most integer `race_id` groups contain one distinct value for each apparently race-level field, but a minority contain conflicting values.

The number of affected `race_id` groups ranges from:

- 92 with multiple `type` values;
- to 210 with multiple `off` or `race_name` values.

**Interpretation**

These conflicts show that `race_id` cannot yet be accepted as a globally reliable standalone race key.

Possible explanations include:

- reuse of the same identifier for different races;
- duplicated or overlapping source records;
- import artefacts;
- amended versions of race information;
- incorrect assumptions about which fields should remain constant within a race.

No key, deduplication or correction rule is selected at this stage.

In [17]:
# Add the total number of integer race_id groups to each row so that the
# absolute conflict counts can also be expressed as percentages.
total_integer_race_ids = len(race_level_consistency)

consistency_summary["total_integer_race_ids"] = total_integer_race_ids

# Calculate the percentage of race_id groups containing more than one distinct
# value for each apparently race-level field.
consistency_summary["percentage_of_race_ids"] = (
    consistency_summary["race_ids_with_multiple_values"]
    / consistency_summary["total_integer_race_ids"]
    * 100
).round(4)

# Show the largest observed inconsistency rates first.
consistency_summary = consistency_summary.sort_values(
    "race_ids_with_multiple_values",
    ascending=False,
).reset_index(drop=True)

display(consistency_summary)

,field,race_ids_with_multiple_values,total_integer_race_ids,percentage_of_race_ids
0,off_times,210,188782,0.1112
1,race_names,210,188782,0.1112
2,courses,207,188782,0.1097
3,dates,206,188782,0.1091
4,distances,198,188782,0.1049
5,ran_values,195,188782,0.1033
6,going_values,179,188782,0.0948
7,classes,166,188782,0.0879
8,types,92,188782,0.0487


### Scale of within-`race_id` inconsistencies

**Profiling evidence**

Within-group conflicts affect a small minority of the 188,782 integer `race_id` values.

The observed rates range from:

- 0.0487% for `type`;
- to 0.1112% for `off` and `race_name`.

**Interpretation**

The low percentages suggest that `race_id` usually behaves like a race identifier.

However, even a small number of conflicts matters when assessing candidate keys. A valid key must identify records consistently across the complete source, not merely in most cases.

The next step is to inspect examples of the affected `race_id` groups before forming a hypothesis about the cause.

In [18]:
# List the columns that count distinct values for fields expected to describe
# the race rather than an individual runner.
consistency_fields = [
    "distinct_dates",
    "distinct_courses",
    "distinct_off_times",
    "distinct_race_names",
    "distinct_types",
    "distinct_classes",
    "distinct_distances",
    "distinct_going_values",
    "distinct_ran_values",
]

# Count how many apparently race-level fields conflict within each race_id.
#
# This score is used only to rank groups for manual inspection. It is not a
# final data-quality severity measure.
race_level_consistency["inconsistent_field_count"] = sum(
    race_level_consistency[column].gt(1).astype(int)
    for column in consistency_fields
)

# Show the ten race_id groups with the greatest number of conflicting fields.
# Larger groups are shown first when the conflict count is tied.
most_inconsistent_race_ids = (
    race_level_consistency
    .sort_values(
        ["inconsistent_field_count", "row_count"],
        ascending=[False, False],
    )
    .head(10)
    .reset_index(drop=True)
)

display(most_inconsistent_race_ids)

,race_id,row_count,distinct_dates,distinct_courses,distinct_off_times,distinct_race_names,distinct_types,distinct_classes,distinct_distances,distinct_going_values,distinct_ran_values,inconsistent_field_count
0,630000,57,4,5,5,5,2,4,4,4,4,9
1,620000,45,3,4,3,4,2,2,4,4,3,9
2,739000,43,4,4,4,4,4,3,4,3,4,9
3,740000,38,3,3,3,3,2,2,3,2,2,9
4,649000,36,3,3,3,3,3,2,3,3,3,9
5,650007,36,3,3,3,3,2,2,2,3,3,9
6,653000,35,3,3,3,3,2,3,3,2,3,9
7,883000,35,3,3,3,3,2,2,3,3,3,9
8,720000,32,2,2,2,2,2,2,2,2,2,9
9,659000,30,3,3,3,3,2,3,3,2,2,9


In [19]:
# Inspect one highly inconsistent race_id in full race-level context.
#
# Only the fields needed to distinguish separate races are selected here.
# Runner-level rows are retained so we can see how many participants belong
# to each apparent race represented by the same identifier.
race_id_example = 630000

race_id_example_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        class,
        dist,
        going,
        ran,
        horse,
        pos
    FROM "{quoted_table_name}"
    WHERE race_id = ?
    ORDER BY
        date,
        course,
        off,
        rowid;
    """,
    connection,
    params=(race_id_example,),
)

display(race_id_example_rows)

,source_rowid,date,course,race_id,off,race_name,type,class,dist,going,ran,horse,pos
0,77168,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Arvios (GB),12
1,77170,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Mega Bowl (FR),11
2,77199,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Atlantic Storm (FR),10
3,77201,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Publio (IRE),8
4,77224,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,True Reflection (IRE),9
5,77231,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Shendini (IRE),6
6,77232,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Lyavenita (FR),5
7,77234,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Angel Royal (FR),4
8,77235,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Anantapur (FR),3
9,77236,2015-06-28,Saint-Cloud (FR),630000,5:20,Prix de Pau (Handicap) (3yo) (Turf),Flat,,1m,Good,12,Prince Du Goyen (FR),2


### Example of `race_id` reuse

**Profiling evidence**

The value `race_id = 630000` is associated with five distinct races:

- Saint-Cloud on 28 June 2015;
- Cartmel on 18 July 2015;
- Newmarket (July) on 18 July 2015;
- Ascot on 24 July 2015;
- Maisons-Laffitte on 6 October 2015.

These groups differ in date, course, off time, race name, race type, class, distance, going and declared field size.

**Conclusion**

`race_id` is not globally unique within the supplied source.

It may still be useful as part of a composite identifier or as a source attribute, but it cannot be accepted as a standalone race key.

The source does not declare a valid race key, and one must not be invented until broader candidate-key testing has been completed.

In [20]:
# Identify race_id values that are associated with more than one distinct date.
#
# Distinct date is used here as the first indicator of identifier reuse because
# one race cannot legitimately occur on multiple calendar dates.
race_ids_with_multiple_dates = pd.read_sql_query(
    f"""
    SELECT
        race_id,
        COUNT(DISTINCT date) AS distinct_dates
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer'
    GROUP BY race_id
    HAVING COUNT(DISTINCT date) > 1;
    """,
    connection,
)

# Retrieve the years in which those reused race_id values occur.
#
# substr(date, 1, 4) is used only for profiling because the observed date values
# are currently stored as text in ISO-like YYYY-MM-DD form. Date validity will
# be tested separately later.
race_id_conflict_years = pd.read_sql_query(
    f"""
    SELECT
        substr(date, 1, 4) AS year,
        COUNT(*) AS affected_rows,
        COUNT(DISTINCT race_id) AS reused_race_ids
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer'
      AND race_id IN (
          SELECT race_id
          FROM "{quoted_table_name}"
          WHERE typeof(race_id) = 'integer'
          GROUP BY race_id
          HAVING COUNT(DISTINCT date) > 1
      )
    GROUP BY substr(date, 1, 4)
    ORDER BY year;
    """,
    connection,
)

display(race_id_conflict_years)

,year,affected_rows,reused_race_ids
0,2015,1137,58
1,2016,1274,59
2,2018,7,1
3,2019,951,47
4,2022,540,28
5,2024,446,20
6,2025,16,1


### Years containing reused `race_id` values

**Profiling evidence**

`race_id` values associated with more than one distinct date occur in:

- 2015;
- 2016;
- 2018;
- 2019;
- 2022;
- 2024;
- 2025.

The largest affected row counts are in 2016, 2015 and 2019.

**Interpretation**

The reuse of `race_id` is not limited to the earliest part of the source.

However, it is also not distributed evenly across all years. This may indicate source-specific import or identifier-generation issues affecting particular periods rather than one consistent rule across the full dataset.

A later candidate-key study should test combinations such as date, course, off time and `race_id`, rather than relying on `race_id` alone.

In [22]:
# Count distinct apparent races for each integer race_id by combining fields
# that should normally identify one scheduled race occurrence.
#
# This is a profiling construct only. It does not yet define the project's
# final race key.
reused_race_id_summary = pd.read_sql_query(
    f"""
    SELECT
        race_id,
        COUNT(*) AS row_count,
        COUNT(
            DISTINCT
            COALESCE(CAST(date AS TEXT), '<NULL>') || '|' ||
            COALESCE(course, '<NULL>') || '|' ||
            COALESCE(off, '<NULL>') || '|' ||
            COALESCE(race_name, '<NULL>')
        ) AS apparent_race_count
    FROM "{quoted_table_name}"
    WHERE typeof(race_id) = 'integer'
    GROUP BY race_id
    HAVING apparent_race_count > 1
    ORDER BY apparent_race_count DESC, row_count DESC;
    """,
    connection,
)

display(reused_race_id_summary.head(20))

,race_id,row_count,apparent_race_count
0,630000,57,5
1,650000,45,5
2,620000,45,4
3,739000,43,4
4,629000,37,4
5,650005,41,3
6,740000,38,3
7,647000,37,3
8,649000,36,3
9,650007,36,3


### Scale of `race_id` reuse

**Profiling evidence**

Some integer `race_id` values are associated with several distinct apparent races when grouped by:

- date;
- course;
- off time;
- race name.

The largest observed examples are:

- `630000`: five apparent races;
- `650000`: five apparent races;
- `620000`: four apparent races;
- `739000`: four apparent races;
- `629000`: four apparent races.

**Interpretation**

The conflicts are not limited to one exceptional identifier.

`race_id` appears to be useful for grouping runners within many races, but it is not globally unique across the complete source.

The date, course, off-time and race-name combination used here is only a profiling aid. It has not been accepted as the final race key.

In [23]:
# Count distinct apparent races using a provisional combination of fields that
# should normally describe one scheduled race occurrence.
#
# This is used only to estimate table grain during profiling. It is not being
# adopted as the final race key.
apparent_race_count = connection.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            date,
            course,
            off,
            race_name
        FROM "{quoted_table_name}"
        WHERE rowid <> 1
        GROUP BY
            date,
            course,
            off,
            race_name
    );
    """
).fetchone()[0]

# Compare the total number of data-like rows with the provisional race count.
# If each row represents one runner, the ratio should resemble a plausible
# average field size.
apparent_grain_summary = pd.DataFrame(
    [
        {
            "metric": "Data-like rows",
            "value": row_count - 1,
        },
        {
            "metric": "Distinct apparent races",
            "value": apparent_race_count,
        },
        {
            "metric": "Mean rows per apparent race",
            "value": round(
                (row_count - 1) / apparent_race_count,
                2,
            ),
        },
    ]
)

display(apparent_grain_summary)

,metric,value
0,Data-like rows,1851285.00
1,Distinct apparent races,189043.00
2,Mean rows per apparent race,9.79


### Initial table-grain finding

**Profiling evidence**

After excluding the imported header row, the source contains:

- 1,851,285 data-like rows;
- 189,043 distinct apparent races under the provisional combination of date, course, off time and race name;
- a mean of 9.79 rows per apparent race.

**Initial conclusion**

The evidence strongly supports a grain of approximately one row per runner in one race.

Race-level fields are therefore repeated across the rows representing the runners in that race.

This remains an initial grain conclusion rather than a final relational design decision. The provisional race grouping has not yet been validated as a unique and complete race key.

### Meaning of table grain

In database profiling, **grain** means the real-world unit represented by one row.

For this source, the current evidence suggests:

> One row represents one horse participating in one race.

Under that grain:

- race-level fields repeat across all runners in the race;
- runner-level fields vary between rows;
- the number of rows for a race should usually resemble its field size.

This is an evidence-based structural interpretation, not yet a final schema decision.

In [24]:
# Inspect the SQLite storage classes used in the date column before calculating
# minimum and maximum dates.
#
# The column is declared NUMERIC, but the source may contain text values because
# the table is not STRICT and already contains one imported header row.
date_storage_types = pd.read_sql_query(
    f"""
    SELECT
        typeof(date) AS sqlite_storage_type,
        COUNT(*) AS row_count
    FROM "{quoted_table_name}"
    GROUP BY typeof(date)
    ORDER BY row_count DESC;
    """,
    connection,
)

display(date_storage_types)

,sqlite_storage_type,row_count
0,text,1851286


In [25]:
# Inspect the range and basic format of values stored in the date column.
#
# The known imported header row is excluded so that the literal value "date"
# does not distort the minimum, maximum or format checks.
date_profile = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS data_like_rows,
        COUNT(date) AS non_null_dates,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_dates,
        COUNT(DISTINCT date) AS distinct_date_values,
        MIN(date) AS minimum_date_value,
        MAX(date) AS maximum_date_value,
        SUM(
            CASE
                WHEN date GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
                THEN 1
                ELSE 0
            END
        ) AS values_matching_yyyy_mm_dd
    FROM "{quoted_table_name}"
    WHERE rowid <> 1;
    """,
    connection,
)

display(date_profile)

,data_like_rows,non_null_dates,null_dates,distinct_date_values,minimum_date_value,maximum_date_value,values_matching_yyyy_mm_dd
0,1851285,1851285,0,4130,2015-01-01,2026-05-27,1851285


### Approximate date coverage

**Profiling evidence**

After excluding the imported header row:

- all 1,851,285 data-like rows contain a non-null `date` value;
- all values match the basic `YYYY-MM-DD` text pattern;
- there are 4,130 distinct date values;
- the minimum observed value is `2015-01-01`;
- the maximum observed value is `2026-05-27`.

**Interpretation**

The supplied `raceform.db` extends into 2026, despite the Kaggle dataset title referring to 2015–2025.

The pattern check confirms consistent text formatting, but it does not yet prove that every value is a valid calendar date. Full date validation belongs in a later data-quality profile.

In [26]:
# Summarise the number of runner rows and provisional race groups by year.
#
# The apparent race count uses the same temporary grouping of date, course,
# off time and race name used earlier. It remains a profiling aid rather than
# an accepted final race key.
yearly_coverage = pd.read_sql_query(
    f"""
    SELECT
        substr(date, 1, 4) AS year,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
            COALESCE(CAST(date AS TEXT), '<NULL>') || '|' ||
            COALESCE(course, '<NULL>') || '|' ||
            COALESCE(off, '<NULL>') || '|' ||
            COALESCE(race_name, '<NULL>')
        ) AS apparent_races
    FROM "{quoted_table_name}"
    WHERE rowid <> 1
    GROUP BY substr(date, 1, 4)
    ORDER BY year;
    """,
    connection,
)

# Add the average number of runner rows per provisional race group.
yearly_coverage["mean_rows_per_apparent_race"] = (
    yearly_coverage["runner_rows"]
    / yearly_coverage["apparent_races"]
).round(2)

display(yearly_coverage)

,year,runner_rows,apparent_races,mean_rows_per_apparent_race
0,2015,157683,16609,9.49
1,2016,157938,16235,9.73
2,2017,167494,17110,9.79
3,2018,172047,17546,9.81
4,2019,173039,17345,9.98
5,2020,121683,11875,10.25
6,2021,175552,17706,9.91
7,2022,167479,17350,9.65
8,2023,156075,16119,9.68
9,2024,169702,17170,9.88


### Annual coverage findings

**Profiling evidence**

The source contains records for every year from 2015 through 2026.

For the complete-looking years, annual volume is generally around:

- 156,000 to 175,000 runner rows;
- 16,000 to 17,700 apparent races.

The mean number of rows per apparent race remains relatively stable, ranging from 9.49 to 10.25.

Two years stand out:

- 2020 contains 121,683 runner rows and 11,875 apparent races;
- 2026 contains 64,929 runner rows and 6,684 apparent races.

**Interpretation**

The 2026 volume is consistent with partial-year coverage ending on 27 May.

The lower 2020 volume is an observed coverage difference. Its cause should not be assumed from the data structure alone and may require later contextual or source-specific investigation.

**Initial contextual hypothesis**

The lower 2020 volume is likely related to COVID-19 disruption, including periods when racing was suspended or operated under restrictions.

That explanation is plausible from external historical context, but it is not demonstrated by the source database itself. The structural finding here is simply that 2020 contains materially fewer races and runner rows than surrounding complete years.

In [27]:
# Collect a focused set of SQLite pragmas that describe how the supplied
# database is stored and interpreted.
#
# These values describe the source file and the SQLite connection. They do not
# by themselves establish data quality or racing-domain correctness.
pragma_queries = {
    "page_size_bytes": "PRAGMA page_size;",
    "page_count": "PRAGMA page_count;",
    "freelist_count": "PRAGMA freelist_count;",
    "encoding": "PRAGMA encoding;",
    "schema_version": "PRAGMA schema_version;",
    "user_version": "PRAGMA user_version;",
    "application_id": "PRAGMA application_id;",
    "foreign_keys_enabled": "PRAGMA foreign_keys;",
    "journal_mode": "PRAGMA journal_mode;",
    "auto_vacuum": "PRAGMA auto_vacuum;",
}

pragma_results = []

for pragma_name, pragma_sql in pragma_queries.items():
    # Each selected pragma returns a single value for this database connection.
    pragma_value = connection.execute(pragma_sql).fetchone()[0]

    pragma_results.append(
        {
            "pragma": pragma_name,
            "value": pragma_value,
        }
    )

sqlite_pragmas = pd.DataFrame(pragma_results)

display(sqlite_pragmas)

,pragma,value
0,page_size_bytes,4096
1,page_count,186969
2,freelist_count,0
3,encoding,UTF-8
4,schema_version,1
5,user_version,0
6,application_id,0
7,foreign_keys_enabled,0
8,journal_mode,delete
9,auto_vacuum,0


### SQLite pragma findings

**Source declaration and database metadata**

The supplied database reports:

- page size: 4,096 bytes;
- page count: 186,969;
- freelist pages: 0;
- text encoding: UTF-8;
- schema version: 1;
- user version: 0;
- application ID: 0;
- journal mode: `delete`;
- auto-vacuum: disabled.

The connection reports foreign-key enforcement as disabled.

**Interpretation**

The database does not use SQLite's `user_version` or `application_id` fields to identify an application schema or source release.

The absence of freelist pages is consistent with a compact source file containing no currently unused database pages.

Foreign-key enforcement being disabled has little practical effect here because the source declares no foreign-key constraints.

In [28]:
# Ask SQLite to verify the internal consistency of the database file.
#
# PRAGMA quick_check examines the database structure more thoroughly than the
# earlier header and catalogue checks, while usually running faster than a full
# PRAGMA integrity_check on a large source file.
#
# The database remains open in read-only mode, so this check cannot modify it.
quick_check_result = connection.execute(
    "PRAGMA quick_check;"
).fetchall()

# SQLite returns one row containing "ok" when no structural problems are found.
# If problems exist, it may return several diagnostic rows instead.
quick_check_output = pd.DataFrame(
    quick_check_result,
    columns=["result"],
)

display(quick_check_output)

,result
0,ok


### Database integrity finding

**Profiling evidence**

SQLite's `PRAGMA quick_check` returned:

`ok`

**Conclusion**

The supplied file passes SQLite's non-destructive quick structural integrity check.

This supports the conclusion that `raceform.db` is a readable and internally coherent SQLite database file. It does not validate the correctness, completeness or racing meaning of the stored data.

In [30]:
# Build one UNION ALL query that counts the actual SQLite storage classes used
# in every declared column.
#
# This is important because the table is not STRICT. A column's declared type
# does not guarantee that every stored value uses the corresponding SQLite
# storage class, as already demonstrated by race_id and the imported header row.
storage_type_queries = []

for column_name in declared_columns["column_name"]:
    # Quote each source column safely as a SQLite identifier.
    quoted_column_name = column_name.replace('"', '""')

    storage_type_queries.append(
        f"""
        SELECT
            '{column_name}' AS column_name,
            typeof("{quoted_column_name}") AS sqlite_storage_type,
            COUNT(*) AS row_count
        FROM "{quoted_table_name}"
        GROUP BY typeof("{quoted_column_name}")
        """
    )

# Combine the per-column profiles into one result set.
all_column_storage_types = pd.read_sql_query(
    "\nUNION ALL\n".join(storage_type_queries),
    connection,
)

# Join the actual storage profile back to the declared SQLite types so that
# differences between declaration and stored values are visible together.
all_column_storage_types = all_column_storage_types.merge(
    declared_columns[
        [
            "column_name",
            "declared_type",
        ]
    ],
    on="column_name",
    how="left",
)

all_column_storage_types = all_column_storage_types[
    [
        "column_name",
        "declared_type",
        "sqlite_storage_type",
        "row_count",
    ]
].sort_values(
    [
        "column_name",
        "row_count",
    ],
    ascending=[True, False],
).reset_index(drop=True)

display(all_column_storage_types)

,column_name,declared_type,sqlite_storage_type,row_count
0,age,INTEGER,integer,1851285
1,age,INTEGER,text,1
2,age_band,TEXT,text,1851286
3,btn,NUMERIC,real,1096043
4,btn,NUMERIC,integer,661250
5,btn,NUMERIC,text,93993
6,class,TEXT,text,1851286
7,comment,TEXT,text,1851286
8,course,TEXT,text,1851286
9,dam,TEXT,text,1851286


### Actual SQLite storage-class findings

**Profiling evidence**

The table uses mixed SQLite storage classes in several columns.

Examples include:

- `race_id`, `age` and `ran`, which contain one text value caused by the imported header row;
- `draw`, `num`, `or`, `pos`, `rpr` and `ts`, which contain both integer and text values;
- `btn` and `ovr_btn`, which contain real, integer and text values;
- `prize`, which is declared `INTEGER` but contains text, real and integer values;
- `date`, which is declared `NUMERIC` but is stored entirely as text.

No column shows values stored using SQLite's `null` storage class.

**Interpretation**

The source relies heavily on SQLite's flexible typing. Declared column types cannot be treated as evidence that all stored values are numeric or consistently typed.

The absence of SQLite `NULL` values does not prove that the dataset is complete. Missing or unavailable information may instead be represented by:

- empty strings;
- whitespace;
- textual markers;
- racing-specific symbols;
- other sentinel values.

Those representations require separate value profiling before any missing-data rules are considered.

In [31]:
# Count empty and whitespace-only text values in every source column.
#
# The earlier storage-class profile found no SQLite NULL values. This check
# tests whether missing or unavailable information may instead be represented
# by blank text.
blank_value_queries = []

for column_name in declared_columns["column_name"]:
    # Quote the source column safely as a SQLite identifier.
    quoted_column_name = column_name.replace('"', '""')

    blank_value_queries.append(
        f"""
        SELECT
            '{column_name}' AS column_name,
            SUM(
                CASE
                    WHEN typeof("{quoted_column_name}") = 'text'
                     AND trim("{quoted_column_name}") = ''
                    THEN 1
                    ELSE 0
                END
            ) AS blank_text_rows
        FROM "{quoted_table_name}"
        """
    )

# Combine the per-column counts into one profiling table.
blank_value_profile = pd.read_sql_query(
    "\nUNION ALL\n".join(blank_value_queries),
    connection,
)

# Add percentages so that blank-value prevalence can be compared across
# columns with the same total row count.
blank_value_profile["percentage_of_rows"] = (
    blank_value_profile["blank_text_rows"]
    / row_count
    * 100
).round(2)

# Show only columns containing at least one blank text value, ordered from the
# largest to the smallest affected row count.
blank_value_profile = (
    blank_value_profile.loc[
        blank_value_profile["blank_text_rows"].gt(0)
    ]
    .sort_values(
        "blank_text_rows",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(blank_value_profile)

,column_name,blank_text_rows,percentage_of_rows
0,sex_rest,1612425,87.10
1,pattern,1587927,85.77
2,hg,1122490,60.63
3,rating_band,1081546,58.42
4,prize,839715,45.36
5,class,768544,41.51
6,draw,592778,32.02
7,comment,340394,18.39
8,sp,9097,0.49
9,num,7032,0.38


### Blank-value findings

**Profiling evidence**

Seventeen columns contain empty or whitespace-only text values.

The largest blank-value rates are:

- `sex_rest`: 87.10%;
- `pattern`: 85.77%;
- `hg`: 60.63%;
- `rating_band`: 58.42%;
- `prize`: 45.36%;
- `class`: 41.51%;
- `draw`: 32.02%;
- `comment`: 18.39%.

Several other fields contain relatively small numbers of blanks, including `sp`, `num`, `going`, `owner`, `trainer` and `jockey`.

**Interpretation**

The source does not use SQLite `NULL` consistently for missing or unavailable information. Blank text is a major part of the source representation.

A blank value must not automatically be interpreted as an error. Depending on the field, it may mean:

- not applicable;
- not published;
- unavailable;
- omitted during extraction;
- genuinely missing.

For example, blank `draw` values may be expected for some race types, while blank `pattern` values may indicate that the race has no pattern classification.

The racing meaning of each blank representation requires field-specific investigation before any cleaning or standardisation rule is adopted.

In [32]:
# List the columns that SQLite declares as INTEGER or NUMERIC.
#
# Several of these columns contain text storage values. This check identifies
# the most common text representations without deciding whether they are
# missing-value markers, racing codes or malformed data.
numeric_declared_columns = declared_columns.loc[
    declared_columns["declared_type"].isin(["INTEGER", "NUMERIC"]),
    "column_name",
].tolist()

numeric_text_value_queries = []

for column_name in numeric_declared_columns:
    # Quote each source column safely as a SQLite identifier.
    quoted_column_name = column_name.replace('"', '""')

    numeric_text_value_queries.append(
        f"""
        SELECT
            '{column_name}' AS column_name,
            CAST("{quoted_column_name}" AS TEXT) AS text_value,
            COUNT(*) AS row_count
        FROM "{quoted_table_name}"
        WHERE typeof("{quoted_column_name}") = 'text'
          AND rowid <> 1
        GROUP BY CAST("{quoted_column_name}" AS TEXT)
        """
    )

# Combine the text-value profiles from all numeric-declared columns.
numeric_text_values = pd.read_sql_query(
    "\nUNION ALL\n".join(numeric_text_value_queries),
    connection,
)

# Rank values within each column by frequency so that the most important
# representations can be inspected first.
numeric_text_values["frequency_rank"] = (
    numeric_text_values
    .groupby("column_name")["row_count"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Display the ten most frequent text values from each numeric-declared column.
common_numeric_text_values = (
    numeric_text_values.loc[
        numeric_text_values["frequency_rank"].le(10)
    ]
    .sort_values(
        ["column_name", "frequency_rank"]
    )
    .reset_index(drop=True)
)

display(common_numeric_text_values)

,column_name,text_value,row_count,frequency_rank
0,btn,-,93992,1
1,date,2019-05-11,1094,1
2,date,2022-05-07,1080,2
3,date,2016-06-11,1072,3
4,date,2018-05-05,1065,4
5,date,2025-10-04,1062,5
6,date,2018-10-13,1061,6
7,date,2018-12-26,1059,7
8,date,2022-03-12,1054,8
9,date,2024-05-11,1037,9


### Text representations in numeric-declared columns

**Profiling evidence**

Numeric-declared columns contain several recurring text representations:

- `btn` and `ovr_btn` use `-`;
- `or`, `rpr` and `ts` use an en dash (`–`);
- `draw`, `num` and `prize` contain blank text;
- `pos` contains non-numeric racing outcome codes, including `PU`, `F`, `UR`, `BD`, `DSQ` and others;
- `prize` contains currency-formatted text values such as `€400`;
- `date` is stored entirely as text despite being declared `NUMERIC`.

**Interpretation**

These values should not be treated as one undifferentiated class of invalid numeric data.

They appear to include:

- missing or unavailable markers;
- legitimate racing outcome codes;
- formatted monetary values;
- source typing choices.

Each field will require separate semantic and technical treatment in later profiling and cleaning notebooks.

In [33]:
# Retrieve a small sample of complete source rows from different parts of the
# table.
#
# Using widely separated rowids gives a broader structural view than simply
# selecting the first few records, while still preserving the source exactly
# as stored.
sample_rowids = [
    2,
    row_count // 4,
    row_count // 2,
    (row_count * 3) // 4,
    row_count,
]

sample_rowid_sql = ", ".join(
    str(int(source_rowid))
    for source_rowid in sample_rowids
)

representative_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        *
    FROM "{quoted_table_name}"
    WHERE rowid IN ({sample_rowid_sql})
    ORDER BY rowid;
    """,
    connection,
)

display(representative_rows)

,source_rowid,date,course,race_id,off,race_name,type,class,pattern,rating_band,...,trainer,prize,or,rpr,ts,sire,dam,damsire,owner,comment
0,2,2015-01-01,Catterick,615704,12:30,Happy New Year Novices Hurdle,Hurdle,Class 4,,,...,Brian Ellison,4873.5,–,114,80,Definite Article (GB),The Red Wench (IRE),Aahsaylad,P J Martin,Tracked leaders - effort 3 out - led approachi...
1,462821,2017-11-10,Newcastle (AW),686485,7:45,Sun Bets On The App Handicap (Div I),Flat,Class 6,,0-60,...,Richard Whitaker,,45,27,2,Piccolo (GB),Dahshah GB,Mujtahid,Tony Lumb,Led - headed 6f out - remained prominent - rid...
2,925643,2020-11-10,Newcastle (AW),769502,5:25,Bombardier British Hopped Amber Beer Handicap,Flat,Class 5,,0-70,...,Tim Easterby,300,67,59,17,Holy Roman Emperor (IRE),Galaxie Sud (USA),El Prado,Ryedale Partners No 5,Towards rear - headway 3f out - chased leaders...
3,1388464,2023-08-24,Chepstow,845939,3:55,Blackmore Building Contractors Handicap,Flat,Class 6,,0-65,...,Alexandra Dunn,,63,48,15,Rajasinghe (IRE),Cherry Malotte GB,Pivotal,West Buckland Bloodstock Ltd,Prominent - lost third 2f out - weakened 1f ou...
4,1851286,2026-05-27,Wexford,921545,18:49,Discover Wexford Maiden Hurdle,Hurdle,,,,...,Liam P Cusack,,108,–,–,Soldier Of Fortune (IRE),On Line Tara (IRE),Kayf Tara,John T Murray,In rear - some headway when mistake 3 out - so...


### Representative-row observations

**Profiling evidence**

The sampled records span from the beginning to the end of the source:

- Catterick on 1 January 2015;
- Newcastle in 2017;
- Newcastle in 2020;
- Chepstow in 2023;
- Wexford on 27 May 2026.

Each sampled row combines:

- repeated race information;
- one horse and its connections;
- finishing or performance information;
- ratings;
- pedigree;
- ownership;
- a race comment.

The examples also show several source representations already identified elsewhere:

- blank race-classification fields;
- an en dash in unavailable rating fields;
- blank prize values;
- decimal prize values despite `prize` being declared `INTEGER`;
- race off times represented in both 12-hour and 24-hour-looking formats.

**Interpretation**

The sampled rows are consistent with the proposed grain:

> One row represents one horse participating in one race.

The sample is illustrative only. It does not establish field completeness, validity or typicality.

In [34]:
# Compare the declared field size in `ran` with the number of stored rows in
# each provisional race group.
#
# The provisional group uses date, course, off time and race name. This remains
# a profiling construct rather than an accepted final race key.
ran_reconciliation = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_name,
        COUNT(*) AS stored_runner_rows,
        MIN(ran) AS minimum_ran,
        MAX(ran) AS maximum_ran,
        COUNT(DISTINCT ran) AS distinct_ran_values
    FROM "{quoted_table_name}"
    WHERE rowid <> 1
    GROUP BY
        date,
        course,
        off,
        race_name;
    """,
    connection,
)

# Test whether `ran` is internally constant within the apparent race and
# whether it agrees with the number of stored runner rows.
ran_reconciliation["ran_is_constant"] = (
    ran_reconciliation["distinct_ran_values"].eq(1)
)

ran_reconciliation["ran_matches_stored_rows"] = (
    ran_reconciliation["ran_is_constant"]
    & ran_reconciliation["minimum_ran"].eq(
        ran_reconciliation["stored_runner_rows"]
    )
)

ran_reconciliation_summary = pd.DataFrame(
    [
        {
            "metric": "Apparent races",
            "value": len(ran_reconciliation),
        },
        {
            "metric": "Apparent races with constant ran",
            "value": int(ran_reconciliation["ran_is_constant"].sum()),
        },
        {
            "metric": "Apparent races where ran matches stored rows",
            "value": int(
                ran_reconciliation["ran_matches_stored_rows"].sum()
            ),
        },
        {
            "metric": "Apparent races where ran does not match stored rows",
            "value": int(
                (~ran_reconciliation["ran_matches_stored_rows"]).sum()
            ),
        },
    ]
)

display(ran_reconciliation_summary)

,metric,value
0,Apparent races,189043
1,Apparent races with constant ran,189043
2,Apparent races where ran matches stored rows,189038
3,Apparent races where ran does not match stored...,5


In [35]:
# Select the small number of apparent races where the declared `ran` value
# does not match the number of stored runner rows.
ran_mismatches = (
    ran_reconciliation.loc[
        ~ran_reconciliation["ran_matches_stored_rows"]
    ]
    .copy()
    .sort_values(
        ["date", "course", "off"]
    )
    .reset_index(drop=True)
)

display(ran_mismatches)

,date,course,off,race_name,stored_runner_rows,minimum_ran,maximum_ran,distinct_ran_values,ran_is_constant,ran_matches_stored_rows
0,2024-06-18,Nantes (FR),2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),7,8,8,1,True,False
1,2024-06-26,Ohi (JPN),11:07,Teio Sho (Local (Dirt),4,5,5,1,True,False
2,2024-09-03,Morioka (JPN),11:07,Kozukata Sho (Local (Dirt),4,5,5,1,True,False
3,2024-09-26,Funabashi (JPN),11:07,Marine Cup (Local (Fillies) (Dirt),2,6,6,1,True,False
4,2025-10-09,Ohi (JPN),11:07,Tokyo Hai (Local (Dirt),15,16,16,1,True,False


### `ran` reconciliation findings

**Profiling evidence**

For 189,038 of 189,043 apparent races, the stored runner-row count exactly matches the race-level `ran` value.

Only five apparent races fail this reconciliation:

- Nantes on 18 June 2024: 7 stored rows versus `ran = 8`;
- Ohi on 26 June 2024: 4 stored rows versus `ran = 5`;
- Morioka on 3 September 2024: 4 stored rows versus `ran = 5`;
- Funabashi on 26 September 2024: 2 stored rows versus `ran = 6`;
- Ohi on 9 October 2025: 15 stored rows versus `ran = 16`.

In every exception, `ran` remains constant across the stored rows.

**Conclusion**

The reconciliation provides strong evidence that:

> One source row normally represents one runner in one race.

The five exceptions suggest incomplete runner coverage for a very small number of international races, rather than a general failure of the proposed grain.

No missing runner records will be reconstructed or imputed in this notebook.

In [36]:
# Count the number of distinct stored values in every source column.
#
# This gives a first indication of which fields behave like:
# - low-cardinality classifications;
# - repeated entity names;
# - near-unique descriptive text;
# - possible identifiers.
#
# The counts are descriptive only. A high or low distinct count does not by
# itself establish grain, key validity or racing meaning.
distinct_value_queries = []

for column_name in declared_columns["column_name"]:
    # Quote each source column safely as a SQLite identifier.
    quoted_column_name = column_name.replace('"', '""')

    distinct_value_queries.append(
        f"""
        SELECT
            '{column_name}' AS column_name,
            COUNT(DISTINCT "{quoted_column_name}") AS distinct_values
        FROM "{quoted_table_name}"
        WHERE rowid <> 1
        """
    )

# Combine the per-column counts into one profiling table.
column_cardinality = pd.read_sql_query(
    "\nUNION ALL\n".join(distinct_value_queries),
    connection,
)

# Add the proportion of data-like rows represented by distinct values.
# This helps distinguish highly repeated fields from near-unique fields.
column_cardinality["distinct_percentage_of_rows"] = (
    column_cardinality["distinct_values"]
    / (row_count - 1)
    * 100
).round(4)

# Join the declared type for context and order from highest to lowest
# cardinality.
column_cardinality = (
    column_cardinality
    .merge(
        declared_columns[
            [
                "column_name",
                "declared_type",
            ]
        ],
        on="column_name",
        how="left",
    )
    [
        [
            "column_name",
            "declared_type",
            "distinct_values",
            "distinct_percentage_of_rows",
        ]
    ]
    .sort_values(
        "distinct_values",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(column_cardinality)

,column_name,declared_type,distinct_values,distinct_percentage_of_rows
0,comment,TEXT,1426746,77.0679
1,horse,TEXT,208631,11.2695
2,race_id,INTEGER,188782,10.1973
3,race_name,TEXT,108632,5.8679
4,dam,TEXT,104973,5.6703
5,owner,TEXT,98235,5.3063
6,prize,INTEGER,47215,2.5504
7,time,TEXT,46537,2.5138
8,trainer,TEXT,10709,0.5785
9,jockey,TEXT,7918,0.4277


### Column-cardinality findings

**Profiling evidence**

The columns vary substantially in cardinality.

High-cardinality fields include:

- `comment`: 1,426,746 distinct values;
- `horse`: 208,631;
- `race_id`: 188,782;
- `race_name`: 108,632;
- `dam`: 104,973;
- `owner`: 98,235.

Low-cardinality fields include:

- `type`: 4 distinct values;
- `sex_rest`: 6;
- `class`: 8;
- `sex`: 8;
- `pattern`: 11;
- `age`: 19;
- `going`: 22.

**Initial interpretation**

The profile is consistent with several broad field roles:

- race or runner identifiers and names;
- repeated entities such as horses, trainers, jockeys and owners;
- low-cardinality racing classifications;
- near-free-text descriptions such as `comment`.

Cardinality alone does not prove that repeated names identify the same real-world entity. Name variants, spelling differences and reused names may exist.

Similarly, high cardinality does not make a field a valid key. For example, `race_id` has already been shown to be reused across distinct races.

In [37]:
# Inspect the complete set of stored values for fields with relatively low
# cardinality.
#
# These fields are likely to contain classifications, categories or racing
# codes. Listing their values helps identify which require later domain
# interpretation without yet standardising or correcting them.
low_cardinality_columns = [
    "type",
    "class",
    "pattern",
    "age_band",
    "sex_rest",
    "sex",
    "going",
]

low_cardinality_profiles = {}

for column_name in low_cardinality_columns:
    # Quote the source column safely as a SQLite identifier.
    quoted_column_name = column_name.replace('"', '""')

    value_counts = pd.read_sql_query(
        f"""
        SELECT
            "{quoted_column_name}" AS stored_value,
            COUNT(*) AS row_count
        FROM "{quoted_table_name}"
        WHERE rowid <> 1
        GROUP BY "{quoted_column_name}"
        ORDER BY row_count DESC, stored_value;
        """,
        connection,
    )

    low_cardinality_profiles[column_name] = value_counts

    print(f"{column_name} values")
    display(value_counts)

type values


,stored_value,row_count
0,Flat,1268229
1,Hurdle,358441
2,Chase,179645
3,NH Flat,44970


class values


,stored_value,row_count
0,,768544
1,Class 5,312687
2,Class 4,307371
3,Class 6,191263
4,Class 3,124168
5,Class 2,81783
6,Class 1,63008
7,Class 7,2461


pattern values


...

,stored_value,row_count
0,,1612425
1,F,123667
2,M,52478
3,F & M,37354
4,C & G,25292
5,C & F,69


sex values


,stored_value,row_count
0,G,1078420
1,F,371961
2,M,190797
3,C,178499
4,H,30728
5,R,878
6,B,1
7,BB,1


going values


,stored_value,row_count
0,Good,499160
1,Standard,343818
2,Soft,211462
3,Good To Soft,184351
4,Good To Firm,177363
5,Heavy,116237
6,Standard To Slow,62870
7,Yielding,48747
8,Fast,45728
9,Very Soft,43035


### Low-cardinality field findings

**Profiling evidence**

The `type` field contains four values:

- `Flat`;
- `Hurdle`;
- `Chase`;
- `NH Flat`.

The `class` and `pattern` fields use blank text extensively alongside recognised-looking classifications.

The `age_band` and `sex_rest` fields contain compact eligibility descriptions such as:

- `4yo+`;
- `3-5yo`;
- `F`;
- `F & M`;
- `C & G`.

The runner-level `sex` field contains eight stored values. Most rows use `G`, `F`, `M`, `C` or `H`, while `R`, `B` and `BB` are rare.

The `going` field contains 22 values, including terms associated with different racing jurisdictions and surfaces, such as:

- `Standard`;
- `Yielding`;
- `Very Soft`;
- `Sloppy`;
- `Muddy`;
- `Frozen`.

**Interpretation**

These fields contain meaningful racing categories rather than arbitrary free text, but their codes and jurisdiction-specific terminology require explicit documentation.

Blank `class`, `pattern`, `age_band`, `sex_rest` or `going` values may indicate that the classification is not applicable or was not supplied. They must not be filled automatically.

Rare `sex` values such as `B` and `BB` require direct inspection before they are accepted as valid codes, interpreted as source anomalies or standardised.

In [38]:
# Retrieve every row using the rare sex codes B or BB.
#
# These values occur only once each, so inspecting the complete records is more
# informative than treating them as ordinary categories or automatic errors.
rare_sex_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        horse,
        age,
        sex,
        sire,
        dam,
        trainer,
        owner
    FROM "{quoted_table_name}"
    WHERE rowid <> 1
      AND sex IN ('B', 'BB')
    ORDER BY rowid;
    """,
    connection,
)

display(rare_sex_rows)

,source_rowid,date,course,race_id,off,race_name,type,horse,age,sex,sire,dam,trainer,owner
0,448648,2017-10-15,Cologne (GER),687124,1:35,Kolner Steher Cup Der Pferdeklinik Burg () (3y...,Flat,Par Coeur (GER),3,BB,Adlerflug (GER),Palucca (GER),W Mongil,Dirk Von Mitzlaff
1,814999,2019-11-29,Gulfstream Park (USA),746195,8:30,Starter Optional Claiming (Claimer) (2yo Filli...,Flat,La Venezolana (VEN),2,B,Vacation (USA),Gran Estefania (VEN),Rodolfo Garcia,Paumar Racing Stable Llc


### Rare `sex` code findings

**Profiling evidence**

The two rare runner-sex values occur in international Flat-racing records:

- `BB`: Par Coeur at Cologne, Germany, on 15 October 2017;
- `B`: La Venezolana at Gulfstream Park, United States, on 29 November 2019.

**Interpretation**

The values may be jurisdiction-specific sex descriptions or source abbreviations rather than simple data-entry errors.

Their precise meaning is not established by the database itself. They should remain unchanged until checked against reliable racing-domain documentation or the original source representation.

In [39]:
# Summarise the most common course values in the source.
#
# Course names may include jurisdiction markers or track variants such as
# "(AW)", "(FR)" or "(USA)". This profile records the stored labels without
# attempting to standardise them.
course_profile = pd.read_sql_query(
    f"""
    SELECT
        course,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
            COALESCE(CAST(date AS TEXT), '<NULL>') || '|' ||
            COALESCE(off, '<NULL>') || '|' ||
            COALESCE(race_name, '<NULL>')
        ) AS apparent_races
    FROM "{quoted_table_name}"
    WHERE rowid <> 1
    GROUP BY course
    ORDER BY runner_rows DESC, course
    LIMIT 25;
    """,
    connection,
)

display(course_profile)

,course,runner_rows,apparent_races
0,Wolverhampton (AW),64746,7042
1,Sha Tin (HK),52010,4160
2,Kempton (AW),50268,5046
3,Chantilly (FR),48744,4140
4,Deauville (FR),48618,4051
5,Newcastle (AW),42092,4409
6,Lingfield (AW),41632,4803
7,Dundalk (AW) (IRE),39075,3318
8,Chelmsford (AW),37741,4310
9,Southwell (AW),34488,3816


### Course-coverage findings

**Profiling evidence**

The source contains 528 distinct stored course values.

Among the 25 courses with the most runner rows are tracks in:

- Great Britain;
- Ireland;
- France;
- Hong Kong.

Examples include:

- Wolverhampton, Kempton and Newcastle;
- Dundalk, Curragh and Leopardstown;
- Chantilly, Deauville and Auteuil;
- Sha Tin and Happy Valley.

Several course labels also contain apparent surface or jurisdiction markers, including:

- `(AW)`;
- `(IRE)`;
- `(FR)`;
- `(HK)`.

**Conclusion**

The supplied `raceform.db` is not restricted to UK and Irish racing.

It contains substantial international racing data, so the Kaggle dataset title does not fully describe the geographical scope of this source product.

Course labels should be preserved as supplied until track identity, jurisdiction and surface notation are profiled more systematically.

In [40]:
# Extract parenthesised suffixes from course names as a first-pass indicator of
# jurisdiction or track type.
#
# This is only a structural profile. A suffix such as "(AW)" may describe the
# racing surface rather than the country, while some courses may contain more
# than one parenthesised marker.
course_suffix_profile = pd.read_sql_query(
    f"""
    SELECT
        CASE
            WHEN instr(course, '(') > 0
            THEN substr(course, instr(course, '('))
            ELSE '<no suffix>'
        END AS course_suffix,
        COUNT(DISTINCT course) AS distinct_courses,
        COUNT(*) AS runner_rows
    FROM "{quoted_table_name}"
    WHERE rowid <> 1
    GROUP BY
        CASE
            WHEN instr(course, '(') > 0
            THEN substr(course, instr(course, '('))
            ELSE '<no suffix>'
        END
    ORDER BY runner_rows DESC, course_suffix;
    """,
    connection,
)

display(course_suffix_profile)

,course_suffix,distinct_courses,runner_rows
0,<no suffix>,189,749748
1,(IRE),25,299546
2,(AW),7,274133
3,(FR),73,214637
4,(HK),2,83068
5,(USA),55,48156
6,(AUS),50,42879
7,(AW) (IRE),1,39075
8,(UAE),5,25178
9,(JPN),21,21202


### Course-suffix findings

**Profiling evidence**

Parenthesised course-name suffixes indicate coverage across many international jurisdictions, including:

- Ireland;
- France;
- Hong Kong;
- the United States;
- Australia;
- the United Arab Emirates;
- Japan;
- Germany;
- South Africa;
- and several others.

Large row counts are associated with markers such as:

- `(IRE)`;
- `(FR)`;
- `(HK)`;
- `(USA)`;
- `(AUS)`.

**Limitation**

The parenthesised suffix is not consistently a jurisdiction code.

Examples include:

- `(AW)`, which appears to describe an all-weather course;
- `(July)`, which distinguishes Newmarket's July Course;
- `(Perth)`;
- `(South Carolina)`;
- combined values such as `(AW) (IRE)`.

Therefore, extracting the final parenthesised text is not sufficient to determine a course's country or surface reliably.

**Initial conclusion**

Course identity, jurisdiction and surface are combined within one source text field using inconsistent naming conventions.

These components will require a documented reference mapping or more careful parsing in a later modelling stage. The original `course` value must remain preserved.

In [42]:
# Summarise how often the most common horses, trainers, jockeys and owners
# appear in the source.
#
# Repeated names may represent recurring real-world entities, but name strings
# alone are not yet accepted as stable identifiers.
entity_columns = [
    "horse",
    "trainer",
    "jockey",
    "owner",
]

entity_frequency_profiles = {}

for column_name in entity_columns:
    # Quote the source column safely as a SQLite identifier.
    quoted_column_name = column_name.replace('"', '""')

    entity_frequency = pd.read_sql_query(
        f"""
        SELECT
            "{quoted_column_name}" AS stored_name,
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT date) AS distinct_dates
        FROM "{quoted_table_name}"
        WHERE rowid <> 1
          AND trim("{quoted_column_name}") <> ''
        GROUP BY "{quoted_column_name}"
        ORDER BY runner_rows DESC, stored_name
        LIMIT 10;
        """,
        connection,
    )

    entity_frequency_profiles[column_name] = entity_frequency

    print(f"Most frequent {column_name} values")
    display(entity_frequency)

Most frequent horse values


,stored_name,runner_rows,distinct_dates
0,Red Stripes (USA),177,177
1,Peachey Carnehan (GB),166,166
2,Rockley Point (GB),152,152
3,Geological (IRE),144,144
4,Harbour Vision (GB),144,144
5,Athollblair Boy (IRE),135,135
6,Air Of York (IRE),132,132
7,Tathmeen (IRE),131,131
8,Uther Pendragon (IRE),131,131
9,Muscika (GB),129,129


Most frequent trainer values


,stored_name,runner_rows,distinct_dates
0,Gordon Elliott,15140,2545
1,Richard Fahey,14643,2924
2,Tim Easterby,13367,2603
3,Richard Hannon,13165,2713
4,Joseph Patrick OBrien,11068,2510
5,David OMeara,10847,2948
6,W P Mullins,10807,2287
7,Andrew Balding,9804,2833
8,Michael Appleby,9584,3051
9,Tony Carroll,9521,2813


Most frequent jockey values


,stored_name,runner_rows,distinct_dates
0,Luke Morris,14241,3154
1,David Probert,10128,2638
2,Oisin Murphy,10017,2200
3,Tom Marquand,9918,2586
4,Maxime Guyon,9036,1907
5,Brian Hughes,8922,2386
6,Hollie Doyle,8594,2469
7,Silvestre De Sousa,8382,1926
8,Colin Keane,8350,1789
9,Mickael Barzalona,8024,1849


Most frequent owner values


,stored_name,runner_rows,distinct_dates
0,John P Mcmanus,15384,3206
1,Godolphin,13416,2889
2,Gigginstown House Stud,6998,1764
3,Hamdan Al Maktoum,6867,1565
4,Sheikh Hamdan Bin Mohammed Al Maktoum,4539,1920
5,Mrs J S Bolger,3779,1408
6,Simon Munir Isaac Souede,3410,1766
7,Cheveley Park Stud,3283,1852
8,H H Aga Khan,2970,1493
9,Wertheimer Frere,2797,1431


### Horse-name repetition findings

**Profiling evidence**

The most frequently occurring horse-name values appear between 129 and 177 times.

For each of the ten most frequent names, the number of runner rows equals the number of distinct dates.

**Initial interpretation**

The repeated horse names are consistent with individual horses making multiple race appearances over time.

The equality between runner rows and distinct dates also indicates that these highly frequent horses do not appear more than once on the same date in this profile.

However, the `horse` field is a name string rather than a declared identifier. It may contain:

- spelling variants;
- punctuation differences;
- country suffix differences;
- separate horses sharing the same name;
- changes or corrections over time.

Horse names must therefore not be accepted automatically as unique entity keys.

In [43]:
# Display the trainer profile already created in Cell 69.
#
# Trainer names repeat across many runner records because one trainer can have
# multiple horses running over many dates. The stored name is still only a text
# label and has not been accepted as a stable trainer identifier.
display(entity_frequency_profiles["trainer"])

,stored_name,runner_rows,distinct_dates
0,Gordon Elliott,15140,2545
1,Richard Fahey,14643,2924
2,Tim Easterby,13367,2603
3,Richard Hannon,13165,2713
4,Joseph Patrick OBrien,11068,2510
5,David OMeara,10847,2948
6,W P Mullins,10807,2287
7,Andrew Balding,9804,2833
8,Michael Appleby,9584,3051
9,Tony Carroll,9521,2813


### Trainer-name repetition findings

**Profiling evidence**

The most frequent trainer-name values occur across thousands of dates and many thousands of runner rows.

For example:

- Gordon Elliott appears in 15,140 runner rows across 2,545 dates;
- Richard Fahey appears in 14,643 rows across 2,924 dates;
- Tim Easterby appears in 13,367 rows across 2,603 dates.

**Interpretation**

These patterns are consistent with trainers being recurring entities associated with multiple runners, including multiple runners on the same date.

However, `trainer` is stored only as text. The source declares no trainer identifier, so name strings cannot yet be assumed to provide reliable entity identity.

Potential issues include:

- spelling and punctuation variants;
- initials versus full names;
- name changes;
- different people sharing similar names;
- jurisdiction-specific formatting.

In [44]:
# Display the jockey profile already created in Cell 69.
#
# Jockey names are expected to repeat across many runner records and dates.
# As with trainer names, the stored text is not yet treated as a stable entity
# identifier.
display(entity_frequency_profiles["jockey"])

,stored_name,runner_rows,distinct_dates
0,Luke Morris,14241,3154
1,David Probert,10128,2638
2,Oisin Murphy,10017,2200
3,Tom Marquand,9918,2586
4,Maxime Guyon,9036,1907
5,Brian Hughes,8922,2386
6,Hollie Doyle,8594,2469
7,Silvestre De Sousa,8382,1926
8,Colin Keane,8350,1789
9,Mickael Barzalona,8024,1849


### Jockey-name repetition findings

**Profiling evidence**

The most frequent jockey-name values occur across thousands of runner rows and race dates.

For example:

- Luke Morris appears in 14,241 runner rows across 3,154 dates;
- David Probert appears in 10,128 rows across 2,638 dates;
- Oisin Murphy appears in 10,017 rows across 2,200 dates.

**Interpretation**

These patterns are consistent with jockeys being recurring entities who ride multiple horses, including multiple rides on the same date.

The `jockey` field is stored only as text, and the source declares no jockey identifier.

Name strings must therefore not yet be treated as stable entity keys. Later investigation should consider spelling, accents, initials, abbreviations, name collisions and changes in source formatting.

In [45]:
# Display the owner profile already created in Cell 69.
#
# Owner names may represent individuals, partnerships, syndicates, companies
# or other ownership structures. The stored text is therefore profiled as a
# repeated label rather than assumed to identify one consistent entity type.
display(entity_frequency_profiles["owner"])

,stored_name,runner_rows,distinct_dates
0,John P Mcmanus,15384,3206
1,Godolphin,13416,2889
2,Gigginstown House Stud,6998,1764
3,Hamdan Al Maktoum,6867,1565
4,Sheikh Hamdan Bin Mohammed Al Maktoum,4539,1920
5,Mrs J S Bolger,3779,1408
6,Simon Munir Isaac Souede,3410,1766
7,Cheveley Park Stud,3283,1852
8,H H Aga Khan,2970,1493
9,Wertheimer Frere,2797,1431


### Owner-name repetition findings

**Profiling evidence**

The most frequent owner values recur across thousands of runner rows and dates.

Examples include:

- John P McManus: 15,384 runner rows across 3,206 dates;
- Godolphin: 13,416 rows across 2,889 dates;
- Gigginstown House Stud: 6,998 rows across 1,764 dates.

**Interpretation**

The `owner` field represents recurring ownership labels, but those labels may refer to different kinds of entities, including:

- individuals;
- families;
- studs;
- companies;
- partnerships;
- syndicates.

The source provides no owner identifier or entity type.

Owner text should therefore be preserved as supplied and not assumed to represent a single, consistently defined entity model.

In [46]:
# Test whether a provisional race description plus horse name identifies one
# runner row uniquely.
#
# This is not a proposed final key. It is a structural test of the likely
# runner-level grain using fields already shown to describe a race and runner.
runner_key_profile = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS data_rows,
        COUNT(
            DISTINCT
            COALESCE(CAST(date AS TEXT), '<NULL>') || '|' ||
            COALESCE(course, '<NULL>') || '|' ||
            COALESCE(off, '<NULL>') || '|' ||
            COALESCE(race_name, '<NULL>') || '|' ||
            COALESCE(horse, '<NULL>')
        ) AS distinct_provisional_runner_keys
    FROM "{quoted_table_name}"
    WHERE rowid <> 1;
    """,
    connection,
)

runner_key_profile["duplicate_rows_under_test"] = (
    runner_key_profile["data_rows"]
    - runner_key_profile["distinct_provisional_runner_keys"]
)

display(runner_key_profile)

,data_rows,distinct_provisional_runner_keys,duplicate_rows_under_test
0,1851285,1851285,0


### Provisional runner-key findings

**Profiling evidence**

The provisional combination of:

- `date`;
- `course`;
- `off`;
- `race_name`;
- `horse`;

produces 1,851,285 distinct values across 1,851,285 data rows.

No duplicate rows were found under this test.

**Interpretation**

This strongly supports the conclusion that the source grain is:

> one row per horse participating in one race.

It also provides a useful provisional way to identify individual runner records during profiling.

However, this is not yet accepted as the final database key. It depends on several descriptive text fields and may be vulnerable to:

- corrected race names;
- course-label changes;
- off-time corrections;
- horse-name formatting changes;
- future source files using different naming conventions.

A durable target model should retain the original source fields while introducing explicit race and runner identifiers through a documented transformation process.

In [47]:
# Test whether the stored runner number is unique within each provisional race.
#
# The provisional race definition uses date, course, off time and race name.
# Blank runner numbers are excluded because they cannot establish uniqueness.
runner_number_profile = pd.read_sql_query(
    f"""
    WITH race_runner_numbers AS (
        SELECT
            date,
            course,
            off,
            race_name,
            num,
            COUNT(*) AS rows_with_number
        FROM "{quoted_table_name}"
        WHERE rowid <> 1
          AND trim(CAST(num AS TEXT)) <> ''
        GROUP BY
            date,
            course,
            off,
            race_name,
            num
    )
    SELECT
        COUNT(*) AS distinct_race_number_combinations,
        SUM(CASE WHEN rows_with_number > 1 THEN 1 ELSE 0 END)
            AS duplicated_numbers_within_race,
        MAX(rows_with_number) AS maximum_rows_for_one_race_number
    FROM race_runner_numbers;
    """,
    connection,
)

display(runner_number_profile)

,distinct_race_number_combinations,duplicated_numbers_within_race,maximum_rows_for_one_race_number
0,1842699,700,17


In [48]:
# Show the largest cases where the same non-blank runner number appears more
# than once within the same provisional race.
#
# This helps distinguish genuine duplicate numbering from source conventions
# such as coupled entries, reserves, bracketed runners or malformed values.
duplicated_runner_numbers = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_name,
        CAST(num AS TEXT) AS runner_number,
        COUNT(*) AS rows_with_number,
        GROUP_CONCAT(horse, ' | ') AS horses
    FROM "{quoted_table_name}"
    WHERE rowid <> 1
      AND trim(CAST(num AS TEXT)) <> ''
    GROUP BY
        date,
        course,
        off,
        race_name,
        CAST(num AS TEXT)
    HAVING COUNT(*) > 1
    ORDER BY
        rows_with_number DESC,
        date,
        course,
        off
    LIMIT 20;
    """,
    connection,
)

display(duplicated_runner_numbers)

,date,course,off,race_name,runner_number,rows_with_number,horses
0,2026-03-21,Chukyo,06:20,Chunichi Sports Sho Falcon Stakes (Turf),0,17,Diamond Knot (JPN) | A Shin Deed (JPN) | Fukuc...
1,2016-12-07,Lyon-La Soie (FR),5:10,Prix Du Haras Des Chataigniers (Claimer) (All-...,0,16,Absolute Summer (FR) | Miss Charlotte (IRE) | ...
2,2018-11-04,Kyoto (JPN),6:01,JBC Sprint (Local (Dirt),0,16,Graceful Leap (JPN) | Matera Sky (USA) | Kitas...
3,2026-03-21,Nakayama,06:45,Flower Cup (Fillies) (Turf),0,16,Smart Priere (JPN) | Longing Celine (JPN) | Ex...
4,2018-08-16,Mombetsu (JPN),11:07,Breeders Gold Cup (Local (Dirt),0,15,Up To You (JPN) | So This Is Love (JPN) | Cros...
5,2015-02-28,Caulfield (AUS),5:35,Crown Golden Ale Peter Young Stakes (Turf),0,13,Mourinho (AUS) | Happy Trails (AUS) | Akzar (I...
6,2015-02-28,Caulfield (AUS),3:40,Polytrack Angus Armanasco Stakes (Fillies) (T...,0,11,Sabatini (AUS) | Fontein Ruby (AUS) | Samartes...
7,2016-05-15,Les Landes (JER),3:05,Bloodstock Advisory Services Handicap,0,11,Pas DAction (GB) | Valmina (GB) | Tax Reform (...
8,2016-10-01,Santa Anita (USA),1:06,Eddie D Stakes (Div II) (3yo+) (Turf),0,11,Holy Lute (USA) | Boozer (USA) | Guns Loaded (...
9,2018-04-22,Les Landes (JER),4:50,The Presidents Handicap,0,11,George Baker (IRE) | Major Tom (GB) | Pas DAct...


In [49]:
# Separate duplicated runner-number groups into:
# 1. groups where the stored number is zero; and
# 2. groups where a positive runner number is repeated.
#
# This tests whether the apparent duplication is mainly caused by zero being
# used as a missing or unavailable value.
runner_number_duplicate_summary = pd.read_sql_query(
    f"""
    WITH duplicated_numbers AS (
        SELECT
            date,
            course,
            off,
            race_name,
            CAST(num AS INTEGER) AS runner_number,
            COUNT(*) AS rows_with_number
        FROM "{quoted_table_name}"
        WHERE rowid <> 1
          AND trim(CAST(num AS TEXT)) <> ''
        GROUP BY
            date,
            course,
            off,
            race_name,
            CAST(num AS INTEGER)
        HAVING COUNT(*) > 1
    )
    SELECT
        CASE
            WHEN runner_number = 0 THEN 'zero'
            ELSE 'positive'
        END AS number_category,
        COUNT(*) AS duplicated_race_number_groups,
        SUM(rows_with_number) AS affected_runner_rows,
        MAX(rows_with_number) AS maximum_rows_in_group
    FROM duplicated_numbers
    GROUP BY
        CASE
            WHEN runner_number = 0 THEN 'zero'
            ELSE 'positive'
        END
    ORDER BY number_category;
    """,
    connection,
)

display(runner_number_duplicate_summary)

,number_category,duplicated_race_number_groups,affected_runner_rows,maximum_rows_in_group
0,positive,523,1084,4
1,zero,177,1170,17


In [50]:
# Inspect the largest cases where a positive runner number is repeated within
# the same provisional race.
#
# Positive duplicates may reveal coupled entries, shared betting numbers,
# reserves, source errors or a different jurisdiction-specific convention.
positive_runner_number_duplicates = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        race_name,
        CAST(num AS INTEGER) AS runner_number,
        COUNT(*) AS rows_with_number,
        GROUP_CONCAT(horse, ' | ') AS horses
    FROM "{quoted_table_name}"
    WHERE rowid <> 1
      AND typeof(num) = 'integer'
      AND num > 0
    GROUP BY
        date,
        course,
        off,
        race_name,
        num
    HAVING COUNT(*) > 1
    ORDER BY
        rows_with_number DESC,
        date,
        course,
        off,
        runner_number
    LIMIT 25;
    """,
    connection,
)

display(positive_runner_number_duplicates)

,date,course,off,race_name,runner_number,rows_with_number,horses
0,2018-12-08,San Isidro (ARG),9:55,Copa de Plata (3yo+ Fillies & Mares) (Turf),11,4,London Show (URU) | Litle Eatly (BRZ) | Mendel...
1,2015-01-11,Monterrico (PER),10:25,Premio Enrique Meiggs (3yo+) (Dirt),10,3,Good Luck Keny (ARG) | Sweet Cazanova (ARG) | ...
2,2015-03-07,Aqueduct (USA),9:50,Gotham Stakes (3yo) (Dirt),1,3,Dontbetwithbruno (USA) | Uninfluenced (USA) | ...
3,2015-07-05,Club Hipico de Santiago (CHI),8:35,Premio Arturo Lyon P (3yo Fillies) (Turf),2,3,Superdotada (CHI) | Kitcat (CHI) | Cimalta (CHI)
4,2015-08-02,Club Hipico de Santiago (CHI),8:35,Premio Polla de Potrancas (3yo Fillies) (Turf),3,3,Cimalta (CHI) | Wapi (CHI) | Kitcat (CHI)
5,2015-09-06,Monterrico (PER),10:50,Premio Polla de Potrillos - Roberto Alvarez Ca...,3,3,Mr Inspiration (ARG) | Abueyo (ARG) | Street L...
6,2015-10-04,Monterrico (PER),10:55,Ricardo Ortiz de Zevallos (3yo) (Dirt),2,3,Street Lolo (ARG) | Indio Dakota (ARG) | Abuey...
7,2015-11-08,Monterrico (PER),10:35,Derby Nacional (3yo) (Dirt),9,3,Mr. Omar (CHI) | La Candy (PER) | Rubirosa (PER)
8,2015-11-08,Monterrico (PER),10:35,Derby Nacional (3yo) (Dirt),10,3,Street Lolo (ARG) | Blue Demon (ARG) | Mr. Leg...
9,2015-11-14,San Isidro (ARG),8:25,Premio Enrique Acebal (3yo Fillies) (Turf),5,3,Quatro Folhas (ARG) | Valencia (ARG) | Feet Fi...


### Runner-number findings

**Profiling evidence**

The `num` field is usually unique within a provisional race, but not universally.

Duplicate groups comprise:

- 177 groups where `num = 0`, affecting 1,170 runner rows;
- 523 groups where a positive number is shared, affecting 1,084 runner rows.

The largest positive-number duplicates occur mainly in international races, including tracks in:

- Argentina;
- Peru;
- Chile;
- Uruguay;
- the United States.

Some races contain three or four horses sharing the same positive number.

**Interpretation**

The `num` field is not a universal runner-level identifier.

Observed patterns suggest that:

- `0` can represent an unavailable or unassigned runner number;
- positive numbers may identify coupled or bracketed betting entries containing multiple horses;
- numbering conventions vary by jurisdiction.

Therefore, `num` should be treated as a source racing or betting attribute rather than as a unique key for a runner.

In [51]:
# Compare how selected fields vary within each provisional race.
#
# A field that has one distinct value per race is likely race-level.
# A field that commonly has several values per race is likely runner-level.
# This is a structural profile only and does not yet define the target schema.
field_variation_profile = pd.read_sql_query(
    f"""
    WITH race_field_counts AS (
        SELECT
            date,
            course,
            off,
            race_name,

            COUNT(DISTINCT type) AS type_values,
            COUNT(DISTINCT class) AS class_values,
            COUNT(DISTINCT dist) AS distance_values,
            COUNT(DISTINCT going) AS going_values,
            COUNT(DISTINCT ran) AS ran_values,

            COUNT(DISTINCT horse) AS horse_values,
            COUNT(DISTINCT jockey) AS jockey_values,
            COUNT(DISTINCT trainer) AS trainer_values,
            COUNT(DISTINCT pos) AS position_values,
            COUNT(DISTINCT sp) AS starting_price_values
        FROM "{quoted_table_name}"
        WHERE rowid <> 1
        GROUP BY
            date,
            course,
            off,
            race_name
    )
    SELECT
        COUNT(*) AS apparent_races,

        SUM(type_values = 1) AS races_with_one_type,
        SUM(class_values = 1) AS races_with_one_class,
        SUM(distance_values = 1) AS races_with_one_distance,
        SUM(going_values = 1) AS races_with_one_going,
        SUM(ran_values = 1) AS races_with_one_ran_value,

        SUM(horse_values > 1) AS races_with_multiple_horses,
        SUM(jockey_values > 1) AS races_with_multiple_jockeys,
        SUM(trainer_values > 1) AS races_with_multiple_trainers,
        SUM(position_values > 1) AS races_with_multiple_positions,
        SUM(starting_price_values > 1) AS races_with_multiple_starting_prices
    FROM race_field_counts;
    """,
    connection,
)

display(field_variation_profile)

,apparent_races,races_with_one_type,races_with_one_class,races_with_one_distance,races_with_one_going,races_with_one_ran_value,races_with_multiple_horses,races_with_multiple_jockeys,races_with_multiple_trainers,races_with_multiple_positions,races_with_multiple_starting_prices
0,189043,189043,189043,189043,189043,189043,189021,189021,189009,189020,188333


### Race-level and runner-level field findings

**Profiling evidence**

Across all 189,043 provisional races, the following fields have exactly one stored value within every race:

- `type`;
- `class`;
- `dist`;
- `going`;
- `ran`.

By contrast, the following fields usually contain multiple values within a race:

- `horse`;
- `jockey`;
- `trainer`;
- `pos`;
- `sp`.

For example, 189,021 provisional races contain multiple horse values, while only 22 do not.

**Interpretation**

This strongly supports a mixed-grain source structure in which each row contains:

- race-level attributes repeated across all runners in the race; and
- runner-level attributes specific to the individual horse record.

The source is therefore structurally denormalised.

The small number of races without multiple runner-level values are consistent with singleton or incomplete race records already identified during profiling.

**Initial modelling implication**

A later target model will likely need to separate:

- one race record per race; and
- one runner record per horse entered or recorded in that race.

No target-schema decision is made in this notebook.

In [52]:
# Create a concise field-investigation register from the structural evidence
# collected so far.
#
# This does not clean or redesign the data. It records why selected source
# fields need additional profiling in later notebooks.
field_investigation_register = pd.DataFrame(
    [
        {
            "field": "race_id",
            "observed_issue": "Reused across different dates, courses and races",
            "later_question": "Can a reliable source race identifier be reconstructed?",
        },
        {
            "field": "course",
            "observed_issue": "Combines course identity, jurisdiction and surface markers",
            "later_question": "How should course, country and surface be separated?",
        },
        {
            "field": "num",
            "observed_issue": "Blank, zero and shared positive values occur",
            "later_question": "Which numbering conventions apply by jurisdiction?",
        },
        {
            "field": "pos",
            "observed_issue": "Contains finishing positions and non-finish codes",
            "later_question": "How should numeric placings and outcome codes be represented?",
        },
        {
            "field": "or",
            "observed_issue": "Contains integers and dash placeholders",
            "later_question": "How should unavailable official ratings be represented?",
        },
        {
            "field": "rpr",
            "observed_issue": "Contains integers and dash placeholders",
            "later_question": "How should unavailable Racing Post Ratings be represented?",
        },
        {
            "field": "ts",
            "observed_issue": "Contains integers and dash placeholders",
            "later_question": "How should unavailable Topspeed ratings be represented?",
        },
        {
            "field": "prize",
            "observed_issue": "Contains blanks, integers, decimals and currency-formatted text",
            "later_question": "Can prize value and currency be parsed reliably?",
        },
        {
            "field": "sp",
            "observed_issue": "Stored as text with several source formats",
            "later_question": "Which starting-price formats and special codes occur?",
        },
        {
            "field": "dist",
            "observed_issue": "Stored as descriptive text",
            "later_question": "Can distance be converted to a consistent numeric measure?",
        },
        {
            "field": "wgt",
            "observed_issue": "Stored as descriptive text",
            "later_question": "Can carried weight be converted to a consistent numeric measure?",
        },
        {
            "field": "time",
            "observed_issue": "Stored as text with many distinct values",
            "later_question": "Which timing formats and unavailable-value conventions occur?",
        },
    ]
)

display(field_investigation_register)

,field,observed_issue,later_question
0,race_id,"Reused across different dates, courses and races",Can a reliable source race identifier be recon...
1,course,"Combines course identity, jurisdiction and sur...","How should course, country and surface be sepa..."
2,num,"Blank, zero and shared positive values occur",Which numbering conventions apply by jurisdict...
3,pos,Contains finishing positions and non-finish codes,How should numeric placings and outcome codes ...
4,or,Contains integers and dash placeholders,How should unavailable official ratings be rep...
5,rpr,Contains integers and dash placeholders,How should unavailable Racing Post Ratings be ...
6,ts,Contains integers and dash placeholders,How should unavailable Topspeed ratings be rep...
7,prize,"Contains blanks, integers, decimals and curren...",Can prize value and currency be parsed reliably?
8,sp,Stored as text with several source formats,Which starting-price formats and special codes...
9,dist,Stored as descriptive text,Can distance be converted to a consistent nume...


### Fields requiring deeper investigation

The structural profile identifies several fields that should be examined in later notebooks before any cleaning or modelling rules are adopted.

Key areas include:

- race identity, because `race_id` is reused across different races;
- course identity, jurisdiction and surface, because these are combined in one text field;
- runner numbering, because `num` may be blank, zero or shared;
- finishing outcomes, because `pos` mixes numeric placings with non-finish codes;
- ratings, because `or`, `rpr` and `ts` mix numeric values with placeholders;
- prize money, because `prize` contains multiple numeric and currency formats;
- starting prices, distances, weights and race times, because these are stored as source text rather than consistently typed measures.

These are recorded as investigation questions rather than cleaning instructions.

The raw source values must remain unchanged until each field has been profiled separately and its semantics are sufficiently understood.